# 🚀 Spaceship Titanic - Kaggle Competition Solution
This notebook builds a strong baseline model using feature engineering and XGBoost.
Target: Predict whether a passenger was **Transported** to another dimension.

In [8]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

## 📂 Load Data

In [9]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
train.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


## 🔧 Feature Engineering

In [10]:
def preprocess(df):
    # Split Cabin
    df[['Deck','CabinNum','Side']] = df['Cabin'].str.split('/', expand=True)

    # Group Size
    df['Group'] = df['PassengerId'].str.split('_').str[0]
    df['GroupSize'] = df.groupby('Group')['Group'].transform('count')

    # Total Spending
    spend_cols = ['RoomService','FoodCourt','ShoppingMall','Spa','VRDeck']
    df['TotalSpend'] = df[spend_cols].sum(axis=1)

    # Fix CryoSleep spending
    df.loc[df['CryoSleep'] == True, spend_cols] = 0

    return df

train = preprocess(train)
test = preprocess(test)

## 🧹 Handle Missing Values

In [11]:
for col in train.columns:
    if train[col].dtype == 'object':
        train[col] = train[col].fillna('Unknown')
        test[col] = test[col].fillna('Unknown')
    else:
        # Only compute median for numeric columns
        if train[col].dtype in ['float64', 'int64', 'float32', 'int32']:
            train[col] = train[col].fillna(train[col].median())
            test[col] = test[col].fillna(test[col].median())

## 🔢 Encoding Categorical Variables

In [14]:
cat_cols = train.select_dtypes(include=['object', 'bool']).columns

encoders = {}
for col in cat_cols:
    # Convert to string to ensure consistency
    train[col] = train[col].astype(str)
    test[col] = test[col].astype(str)
    
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col])
    encoders[col] = le
    # Handle unseen labels in test set
    test[col] = test[col].apply(lambda x: le.transform([x])[0] if x in le.classes_ else -1)

C:\Users\Hackerali\AppData\Local\Temp\ipykernel_1924\72896974.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = train.select_dtypes(include=['object', 'bool']).columns


KeyError: 'Transported'

## 🤖 Train Model

In [15]:
X = train.drop('Transported', axis=1).select_dtypes(include=['int64', 'float64', 'int32', 'float32', 'bool'])
y = train['Transported']

model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X, y)

print('Training Accuracy:', accuracy_score(y, model.predict(X)))

ValueError: Invalid classes inferred from unique values of `y`.  Expected: [0 1], got ['False' 'True']

## 🧪 Predict on Test Set

In [16]:
X_test = test.select_dtypes(include=['int64', 'float64', 'int32', 'float32', 'bool'])
preds = model.predict(X_test)

submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Transported': preds.astype(bool)
})

submission.to_csv('submission.csv', index=False)
submission.head()

NotFittedError: need to call fit or load_model beforehand